In [ ]:
#Modify the path to a directory on your machine
import os
os.environ["CRDS_PATH"] = "/home/dmastro/jwst-crds"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# Packages that allow us to get information about objects:
import asdf
import copy
import shutil

# Numpy library:
import numpy as np

# For downloading data
import requests

# List of possible data quality flags
from jwst.datamodels import dqflags

# The entire calwebb_detector1 pipeline
from jwst.pipeline import calwebb_detector1

# Individual steps that make up calwebb_detector1
from jwst.dq_init import DQInitStep
from jwst.saturation import SaturationStep
from jwst.superbias import SuperBiasStep
from jwst.ipc import IPCStep                                                                                    
from jwst.refpix import RefPixStep                                                                
from jwst.linearity import LinearityStep
from jwst.persistence import PersistenceStep
from jwst.dark_current import DarkCurrentStep
from jwst.jump import JumpStep
from jwst.ramp_fitting import RampFitStep
from jwst import datamodels

import matplotlib.pyplot as plt

from astropy.io import fits

# Plotting tools:
from retired.pipeline1_plotting_tools import download_files, plot_jump, plot_jumps, plot_ramp, plot_ramps, show_image, side_by_side

import jwst
print(jwst.__version__)

from pathlib import Path

In [ ]:
#for item in Path('../data/JWST/3000s_exposure').iterdir():
#    print(item)
#for item in Path('../data/JWST/3000s_exposure_blind').iterdir():
#    print(item)

In [ ]:
# Record:
# The pipeline1 does following steps w/ dark current skipped:
# dq_init, saturation, ipc, superbias, refpix, linearity, persistence(not listed), dark_current(skipped), charge_migration(skipped), jump, ramp_fit, gain (1.0 rescale)

In [ ]:
os.environ["CRDS_PATH"] = "/home/dmastro/jwst-crds"
input_file_category = '3000s_exposure'
data_path = Path('../data/JWST/' + input_file_category)
count = 0
for item in data_path.iterdir():
    if count == 1:
        break
    input_file_base = item.name
    output_dir = '../data/JWST/' + input_file_category + '/' + input_file_base
    # 检查结果文件是否已经存在：
    #if os.path.exists(output_dir + '/' + input_file_base + '_jumpstep.fits'):
    #    continue
    uncal_file = output_dir + '/' + input_file_base + '_uncal.fits'
    uncal_model = datamodels.open(uncal_file)
    # Using the run() method. Instantiate and set parameters
    dq_init_step = DQInitStep()
    dq_init_step.output_dir = output_dir
    dq_init_step.save_results = False
    # Call the run() method on the uncal file
    dq_init = dq_init_step.run(uncal_file)
    del uncal_model, dq_init_step#, dq_init

    #dq_init_file = output_dir + '/' + input_file_base + '_dqinitstep.fits'
    #dq_init = datamodels.open(dq_init_file)
    # Using the run() method
    saturation_step = SaturationStep()
    saturation_step.output_dir = output_dir
    saturation_step.save_results = False
    # Call using the the output from the previously-run dq_init step
    saturation = saturation_step.run(dq_init)
    del dq_init, saturation_step#, saturation

    #saturation_file = output_dir + '/' + input_file_base + '_saturationstep.fits'
    #saturation = datamodels.open(saturation_file)
    # Using the run() method
    ipc_step = IPCStep()
    ipc_step.output_dir = output_dir
    ipc_step.save_results = False
    # Call using the the output from the previously-run dq_init step
    ipc = ipc_step.run(saturation)
    del saturation, ipc_step#, ipc

    #ipc_file = output_dir + '/' + input_file_base + '_ipcstep.fits'
    #ipc = datamodels.open(ipc_file)
    # Using the run() method
    superbias_step = SuperBiasStep()
    superbias_step.output_dir = output_dir
    superbias_step.save_results = False
    # Call using the the output from the previously-run saturation step
    superbias = superbias_step.run(ipc)
    del ipc, superbias_step#, superbias

    #superbias_file = output_dir + '/' + input_file_base + '_superbiasstep.fits'
    #superbias = datamodels.open(superbias_file)
    # Instantiate and set parameters
    refpix_step = RefPixStep()
    refpix_step.output_dir = output_dir
    refpix_step.save_results = False
    # Call using the saturation instance from the previously-run
    # saturation step
    refpix = refpix_step.run(superbias)
    del superbias, refpix_step#, refpix

    #refpix_file = output_dir + '/' + input_file_base + '_refpixstep.fits'
    #refpix = datamodels.open(refpix_file)
    # Using the run() method
    linearity_step = LinearityStep()
    linearity_step.output_dir = output_dir
    linearity_step.save_results = True
    # Call using the refpix instance from the previously-run
    # refpix step
    linearity = linearity_step.run(refpix)
    del refpix, linearity_step#, linearity

    #linearity_file = output_dir + '/' + input_file_base + '_linearitystep.fits'
    #linearity = datamodels.open(linearity_file)
    # Using the run() method
    persist_step = PersistenceStep()
    persist_step.output_dir = output_dir
    persist_step.save_results = False
    # Call using the refpix instance from the previously-run
    # linearity step
    persist = persist_step.run(linearity)
    del linearity, persist_step#, persist

    #persist_file = output_dir + '/' + input_file_base + '_persistencestep.fits'
    #persist = datamodels.open(persist_file)
    # Using the run() method
    jump_step = JumpStep()
    jump_step.output_dir = output_dir
    jump_step.save_results = True
    #jump_step.rejection_threshold = 5.0
    jump_step.maximum_cores = 'all'
    # Call using the dark instance from the previously-run
    # dark current subtraction step
    jump = jump_step.run(persist)
    del persist, jump_step, jump
    count = 1

In [ ]:
input_file_category = '3000s_exposure_NRS1_extra'
data_path = Path('../data/JWST/' + input_file_category)
for item in data_path.iterdir():
    input_file_base = item.name
    output_dir = '../data/JWST/' + input_file_category + '/' + input_file_base
    # 检查结果文件是否已经存在：
    if os.path.exists(output_dir + '/' + input_file_base + '_jumpstep_5thr.fits'):
        continue
    linearity_file = output_dir + '/' + input_file_base + '_linearitystep.fits'
    linearity = datamodels.open(linearity_file)
    
    jump_step = JumpStep()
    jump_step.output_dir = output_dir
    jump_step.save_results = True
    jump_step.rejection_threshold = 5.0
    jump_step.maximum_cores = 'all'
    jump_step.output_file = input_file_base + '_jumpstep_5thr'

    jump = jump_step.run(linearity)
    del linearity, jump_step, jump

In [ ]:
data_path.iterdir()

In [ ]:
# Using GAIN reference file: /home/hailin/Documents/CRDS/references/jwst/nirspec/jwst_nirspec_gain_0020.fits
gain_0020 = '/home/hailin/Documents/CRDS/references/jwst/nirspec/jwst_nirspec_gain_0020.fits'
read_0042 = '/home/hailin/Documents/CRDS/references/jwst/nirspec/jwst_nirspec_readnoise_0042.fits'
gain_data = fits.getdata(gain_0020, 'SCI')[4:-4,4:-4]
read_data = fits.getdata(read_0042, 'SCI')

In [ ]:
gain_data.shape

In [ ]:
print(np.max(gain_data))
print(np.min(gain_data))
print(np.average(gain_data))
print(np.std(gain_data))

In [ ]:
selected_gain = gain_data[(gain_data >= 0.6)]
print(np.mean(selected_gain))
print(np.mean(selected_gain))

In [ ]:
print(np.max(read_data))
print(np.min(read_data))
print(np.average(read_data))
print(np.std(read_data))

In [ ]:
plt.hist(read_data.flatten(), bins = np.arange(6,20,0.01))
plt.xlim(6,20)

In [ ]:
plt.hist(gain_data.flatten(), bins = np.arange(0.8,1.4,0.01))
plt.xlim(0.8,1.4)

In [ ]:
primary_headers = fits.getheader(gain_0020,0)
primary_headers

In [ ]:
science_headers = fits.getheader(gain_0020,1)
science_headers

In [ ]:
input_file_category = '3000s_exposure'

input_file_base = 'jw01121008001_02102_00001_nrs2'
output_dir = '../data/JWST/' + input_file_category + '/' + input_file_base

jump_file = output_dir + '/' + input_file_base + '_jumpstep_5thr_jumpstep.fits'
jump = datamodels.open(jump_file)

In [ ]:
# How many total jump flags were added? Note that some pixels
# will have more than one group flagged with a jump.
jump_flags = np.where(jump.groupdq & dqflags.pixel['JUMP_DET'] > 0)
print('{} jump flags detected.'.format(len(jump_flags[0])))

# Create a 4-dimensional map of the jump flags
jump_map = (jump.groupdq & dqflags.pixel['JUMP_DET'] > 0)

# Collapse down to a 2D map of the number of flagged jumps in each pixel
jump_map_2d = np.sum(jump_map[0, :, :, :], axis=0)

# Determine how many pixels have jump flags
jump_map_indexes = np.where(jump_map_2d > 0)
impacted_pix = np.sum(jump_map_2d > 0)
total_pix = 2048 * 2048
print(('{} pixels ({:.2f}% of the detector) have been flagged with '
      'at least one jump.'.format(impacted_pix, 100. * impacted_pix / total_pix)))

# Create an array of group numbers to plot against
group_indexes = np.arange(jump_map.shape[1]).astype(int)

indexes_to_plot = np.random.randint(0, 10001, size=9)
jump_data = np.zeros((jump.shape[1], len(indexes_to_plot)))
jump_grps = np.zeros((jump.shape[1], len(indexes_to_plot))).astype(bool)
jump_locs = []
for counter, idx in enumerate(indexes_to_plot):
    #integ, grp, y, x = jump_flags[idx]
    y = jump_map_indexes[0][idx]
    x = jump_map_indexes[1][idx]
    grp = jump_map[0, :, y, x]

    jump_data[:, counter] = jump.data[0, :, y, x]
    jump_grps[:, counter] = grp
    jump_locs.append((x, y))

plot_jumps(jump_data, jump_grps, jump_locs)

In [ ]:
jump_grps.shape